# One program, three engines

The same `lazyqc` program drives three execution paths: the lazy batch
planner (`plan_all` + `collect`), a hand-lowered Qulacs statevector run,
and a quimb/cotengra tensor contraction. Same physics, three cost
profiles. We check agreement first, then time each path.

In [1]:
import os

# One native thread per process: required before importing Qulacs.
for key in ("QULACS_NUM_THREADS", "OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[key] = "1"

import sys
import time
from pathlib import Path
from statistics import median

ROOT = Path().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "benchmarks"))

import qulacs
from lazyqc import Program, plan_all
from tensor_baseline import prepare as prepare_tensor


def timed(fn, repeats=5):
    """Median wall time in ms plus the last return value."""
    values, stamps = None, []
    for _ in range(repeats):
        start = time.perf_counter_ns()
        values = fn()
        stamps.append((time.perf_counter_ns() - start) / 1e6)
    return values, median(stamps)


## The program: two entangled pairs, four requests

`RY(θ)` then `CX` per pair. Requested: $Z_0$, $Z_1$, $X_0X_1$ (pair A)
and $Z_2Z_3$ (pair B). Pair A serves three outputs from one preparation;
pair B serves one.

In [2]:
program = Program(4).ry(0, 0.4).cx(0, 1).ry(2, 0.8).cx(2, 3)
observables = [{0: "Z"}, {1: "Z"}, {0: "X", 1: "X"}, {2: "Z", 3: "Z"}]
requests = [program.expectation(obs) for obs in observables]
print(f"{program.n_qubits} qubits,", f"{len(requests)} requests")
print("expected:", "(0.921061, 0.921061, 0.389418, 1.0)")

4 qubits, 4 requests
expected: (0.921061, 0.921061, 0.389418, 1.0)


## Path A — lazyqc batch planner

Describe first, execute at `collect()`. The planner sees all four
requests before allocating any state, so it can share pair A's
preparation across three outputs.

In [3]:
plan = plan_all(requests, strategy="grouped")
print(plan.explain()["selected_candidate"])  # classical only; no states yet
result = plan.collect()
print("values:", tuple(round(v, 6) for v in result.values))
print("evolutions:", result.stats.evolutions)
print("max state qubits:", result.stats.max_state_qubits)
print("payload bytes:", result.stats.estimated_state_bytes)

grouped
values: (0.921061, 0.921061, 0.389418, 1.0)
evolutions: 2
max state qubits: 2
payload bytes: 64


## Path B — direct Qulacs

One full 4-qubit state, all observables read from it. Note the sign
trap: Qulacs rotations use $\exp(+i\theta P/2)$ while the IR uses
$\exp(-i\theta P/2)$, so rotation angles are negated on lowering — the
same conversion `lazyqc`'s own backend applies. Hand-rolled lowering
must get this right; the middleware does it once.

In [4]:
def run_qulacs_direct(program, observables):
    n_qubits = program.n_qubits
    circuit = qulacs.QuantumCircuit(n_qubits)
    for gate in program.resolved_operations():
        qubits = list(gate.qubits)
        if gate.name == "CX":
            circuit.add_CNOT_gate(*qubits)
        elif gate.name in ("H", "X", "Y", "Z"):
            getattr(circuit, f"add_{gate.name}_gate")(qubits[0])
        else:
            getattr(circuit, f"add_{gate.name}_gate")(qubits[0], -gate.angle)
    state = qulacs.QuantumState(n_qubits)
    state.set_zero_state()
    circuit.update_quantum_state(state)
    values = []
    for obs in observables:
        observable = qulacs.Observable(n_qubits)
        observable.add_operator(1.0, " ".join(f"{p} {q}" for q, p in sorted(obs.items())))
        values.append(observable.get_expectation_value(state))
    return tuple(values)


direct = run_qulacs_direct(program, observables)
print("values:", tuple(round(v, 6) for v in direct))

values: (0.921061, 0.921061, 0.389418, 1.0)


## Path C — quimb tensor contraction

`tensor_baseline.prepare` (the B4 method): quimb local expectations with
a seeded cotengra path search and shared contraction intermediates.

In [5]:
tensor_plan = prepare_tensor(requests)
print("contraction widths:", tensor_plan.contraction_width)
tensor_values = tensor_plan.collect()
print("values:", tuple(round(v, 6) for v in tensor_values))

contraction widths: 1.0
values: (0.921061, 0.921061, 0.389418, 1.0)


## Agreement and timing

All three must agree to ~1e-12; then we compare wall time (median of 5).

In [6]:
lazy_values = result.values
print("max |lazyqc - qulacs|:", max(abs(a - b) for a, b in zip(lazy_values, direct)))
print("max |lazyqc - quimb|: ", max(abs(a - b) for a, b in zip(lazy_values, tensor_values)))

(_, lazy_ms) = timed(lambda: plan_all(requests, strategy="grouped").collect())
(_, qulacs_ms) = timed(lambda: run_qulacs_direct(program, observables))
(_, quimb_ms) = timed(lambda: prepare_tensor(requests).collect())
print(f"lazyqc grouped: {lazy_ms:.3f} ms  (2 evolutions, 2 qubits each)")
print(f"qulacs direct:  {qulacs_ms:.3f} ms  (1 evolution, 4 qubits)")
print(f"quimb tensor:   {quimb_ms:.3f} ms  (incl. path search each call)")

max |lazyqc - qulacs|: 1.1102230246251565e-16
max |lazyqc - quimb|:  8.881784197001252e-16
lazyqc grouped: 0.154 ms  (2 evolutions, 2 qubits each)
qulacs direct:  0.040 ms  (1 evolution, 4 qubits)
quimb tensor:   12.993 ms  (incl. path search each call)


## Pros and cons on this tiny circuit

- **Direct Qulacs** is fastest here: one 4-qubit evolution dwarfs any
  planning. Price: manual batching (a loop per observable set), manual
  sign conventions, no payload limits, no sharing analysis.
- **lazyqc** pays planning overhead (~0.05 ms) to discover the shared
  preparation. Price: worth it only once saved work exceeds that overhead
  (crossover sits at 8–12 qubits on the pairs family).
- **quimb** re-searches contraction paths on every call and carries
  tensor-library import weight. Price: preparation dominates small cases;
  it wins where dense states explode but contraction stays narrow.

Same program, three answers that agree — the middleware framing lets the
comparison stay about *execution choice*, not reimplementation.